# EDA & Backtest Review

探索特征、标签分布，并可视化 `outputs/backtest_equity.csv` 中的收益曲线。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

from stock_trend.config import load_config
from stock_trend.dataset import build_labeled_frame, prepare_dataset
from stock_trend.features import get_feature_columns

config = load_config(Path("../configs/default.yaml"))
dataset = prepare_dataset(config)
frame = build_labeled_frame(dataset.ohlcv, config)
feature_cols = get_feature_columns(frame)

print(f"Samples: {len(frame)} | Features: {len(feature_cols)}")
print("Label balance:")
print(frame["label"].value_counts(normalize=True))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(dataset.ohlcv["date"], dataset.ohlcv["close"], label="Close")
axes[0].set_title(f"{config['symbol']} Price")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

if "rsi_14" in frame.columns:
    axes[1].plot(frame["date"], frame["rsi_14"], color="orange", label="RSI")
    axes[1].axhline(70, linestyle="--", color="red", alpha=0.5)
    axes[1].axhline(30, linestyle="--", color="green", alpha=0.5)
    axes[1].set_title("RSI")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
output_dir = Path(config.get("output_dir", "outputs"))
equity_path = output_dir / "backtest_equity.csv"
summary_path = output_dir / "backtest_summary.yaml"

if equity_path.exists():
    equity = pd.read_csv(equity_path, parse_dates=["date"])
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(equity["date"], equity["strategy_equity"], label="Strategy")
    ax.plot(equity["date"], equity["buy_hold_equity"], label="Buy & Hold")
    if "benchmark_equity" in equity.columns:
        ax.plot(equity["date"], equity["benchmark_equity"], label="Benchmark")
    ax.set_title("Equity Curves (Test Period)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Run: python -m stock_trend.backtest --config configs/default.yaml")

In [ ]:
importance_path = output_dir / "feature_importance.csv"
if importance_path.exists():
    importance = pd.read_csv(importance_path)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(importance["feature"], importance["importance"])
    ax.set_title("Feature Importance")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("Run train first to generate feature_importance.csv")

In [ ]:
if summary_path.exists():
    with summary_path.open(encoding="utf-8") as f:
        summary = yaml.safe_load(f)
    print(yaml.safe_dump(summary, allow_unicode=True, sort_keys=False))